<a href="https://colab.research.google.com/github/Reginajose/Observatorio-Clima-Grande-desafio/blob/main/analise_explorat%C3%B3ria.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ANÁLISE EXPLORATÓRIA DO DATA CARD DA CAPAG

# 1. Conectando o Google Drive ao Colab

In [11]:

# 1. Conectar o Google Drive ao Colab
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')

# 2. Definir o caminho do arquivo no seu Drive
caminho_arquivo = '/content/drive/MyDrive/Observatório de Clima-Célula 4./Dataset\'s/capag-municipios-2025.xlsx'



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. lendo as Primeiras Linhas do csv

In [13]:

# 3. Carregar a aba da 'Prévia da CAPAG' lendo o cabeçalho na linha correta (header=2)
# Garantimos que o código IBGE seja lido como texto para manter os 7 dígitos
df_capag = pd.read_excel(
    caminho_arquivo,
    sheet_name='Prévia da CAPAG',
    header=2,
    dtype={'Código Município Completo': str}
)

# Remediar eventual linha de rodapé/total nula
df_capag = df_capag.dropna(subset=['Código Município Completo'])



/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


3. VISÃO GERAL DO DATASET

In [18]:

print("======================================================================")
print(f"• Total de Municípios (Linhas): {df_capag.shape[0]}")
print(f"• Total de Atributos (Colunas): {df_capag.shape[1]}")
print(f"• Chave Primária: 'Código Município Completo' ({df_capag['Código Município Completo'].nunique()} códigos únicos)")

• Total de Municípios (Linhas): 5568
• Total de Atributos (Colunas): 23
• Chave Primária: 'Código Município Completo' (5568 códigos únicos)


4. DIAGNÓSTICO DE QUALIDADE DOS DADOS & VALORES FALTANTES/IRREGULARES

In [17]:

print("======================================================================")
# Tratar strings de notas limpando espaços em branco
df_capag['CAPAG_limpa'] = df_capag['CAPAG'].astype(str).str.strip()

# Distribuição completa de todas as Notas
distribuicao = df_capag['CAPAG_limpa'].value_counts(dropna=False)
print("Distribuição das Notas da CAPAG no Brasil:")
print(distribuicao.to_string())

# Identificação de Municípios sem Nota (n.d. / n.e. / Nulos)
sem_nota = df_capag[df_capag['CAPAG_limpa'].isin(['n.d.', 'n.e.', 'nan', 'None'])]
print(f"\n• Municípios sem Nota Atribuída (n.d. ou n.e.): {len(sem_nota)} ({len(sem_nota)/len(df_capag)*100:.2f}%)")
print("  (Motivo: Não enviaram DCA/RGF/RREO no prazo exigido pelo Tesouro Nacional)")

Distribuição das Notas da CAPAG no Brasil:
CAPAG_limpa
C       2184
A       1190
B        976
n.d.     683
A+       265
n.e.     137
B+       113
D         19
nan        1

• Municípios sem Nota Atribuída (n.d. ou n.e.): 821 (14.74%)
  (Motivo: Não enviaram DCA/RGF/RREO no prazo exigido pelo Tesouro Nacional)


In [19]:

print("======================================================================")
# Municípios Elegíveis a Empréstimos (A+, A, B+, B)
elegiveis_credito = df_capag[df_capag['CAPAG_limpa'].isin(['A+', 'A', 'B+', 'B'])]

# Municípios Frágeis / Vedados de Tomar Empréstimos (Notas C e D)
vulneraveis_cd = df_capag[df_capag['CAPAG_limpa'].isin(['C', 'D'])]

qtd_cd = len(vulneraveis_cd)
pct_cd = (qtd_cd / len(df_capag)) * 100

print(f"• Municípios com Capacidade de Crédito (Notas A+, A, B+, B): {len(elegiveis_credito)} ({len(elegiveis_credito)/len(df_capag)*100:.2f}%)")
print(f"• Municípios em Fragilidade Fiscal (Notas C e D): {qtd_cd} ({pct_cd:.2f}%)")
print("  --> NOTA TÉCNICA PARA O CARLOS: Este grupo de 2.203 municípios possui impedimento legal")
print("      para obter garantia da União para empréstimos bancários. Portanto, necessitam 100%")
print("      de repasses a fundo perdido do Governo Federal para obras de prevenção de riscos.")

• Municípios com Capacidade de Crédito (Notas A+, A, B+, B): 2544 (45.69%)
• Municípios em Fragilidade Fiscal (Notas C e D): 2203 (39.57%)
  --> NOTA TÉCNICA PARA O CARLOS: Este grupo de 2.203 municípios possui impedimento legal
      para obter garantia da União para empréstimos bancários. Portanto, necessitam 100%
      de repasses a fundo perdido do Governo Federal para obras de prevenção de riscos.


_________________Transformando em arquivos separados____________________________

In [20]:
import pandas as pd
import zipfile
from google.colab import files

# 1. Montar a estrutura do Resumo Geral
df_resumo = pd.DataFrame([
    {"secao": "Identificacao", "indicador": "Fonte", "valor": "STN / Siconfi (Tesouro Nacional)", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Ano de Referencia", "valor": "Previa Ano-Base 2025 (Setembro/2026)", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Aba Utilizada", "valor": "Previa da CAPAG", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Granularidade", "valor": "1 linha por Municipio", "percentual": ""},
    {"secao": "Identificacao", "indicador": "Chave Primaria", "valor": "Codigo Municipio Completo (IBGE 7 digitos)", "percentual": ""},
    {"secao": "Qualidade", "indicador": "Total de Municipios Analisados", "valor": "5.568", "percentual": "100.0%"},
    {"secao": "Qualidade", "indicador": "Municipios com Nota Atribuida (A+ ate D)", "valor": "4.747", "percentual": "85.26%"},
    {"secao": "Qualidade", "indicador": "Municipios sem Nota (n.d. / n.e.)", "valor": "820", "percentual": "14.74%"},
    {"secao": "Estrategia", "indicador": "Municipios com Capacidade de Credito (A+, A, B+, B)", "valor": "2.544", "percentual": "45.69%"},
    {"secao": "Estrategia", "indicador": "Municipios com Fragilidade Fiscal (C e D)", "valor": "2.203", "percentual": "39.57%"}
])

# 2. Montar a Distribuição das Notas CAPAG
df_distribuicao = pd.DataFrame([
    {"nota_capag": "C", "classificacao_fiscal": "Fragilidade Fiscal / Vedado Emprestimo", "qtd_municipios": 2184, "percentual_nacional": 39.22},
    {"nota_capag": "A", "classificacao_fiscal": "Boa Saude Fiscal / Elegivel a Credito", "qtd_municipios": 1190, "percentual_nacional": 21.37},
    {"nota_capag": "B", "classificacao_fiscal": "Boa Saude Fiscal / Elegivel a Credito", "qtd_municipios": 976, "percentual_nacional": 17.53},
    {"nota_capag": "n.d.", "classificacao_fiscal": "Nao Disponivel (Pendencia Contabil DCA/RGF)", "qtd_municipios": 683, "percentual_nacional": 12.27},
    {"nota_capag": "A+", "classificacao_fiscal": "Excelente Saude Fiscal / Elegivel a Credito", "qtd_municipios": 265, "percentual_nacional": 4.76},
    {"nota_capag": "n.e.", "classificacao_fiscal": "Nao Elegivel (Pendencia de Envio STN)", "qtd_municipios": 137, "percentual_nacional": 2.46},
    {"nota_capag": "B+", "classificacao_fiscal": "Boa Saude Fiscal / Elegivel a Credito", "qtd_municipios": 113, "percentual_nacional": 2.03},
    {"nota_capag": "D", "classificacao_fiscal": "Alta Fragilidade Fiscal / Vedado Emprestimo", "qtd_municipios": 19, "percentual_nacional": 0.34}
])

# 3. Montar as Limitações e Regra de Negócio
df_limitacoes = pd.DataFrame([
    {"tipo": "limitacao", "item": "820 municipios (14.74%) nao possuem nota oficial (n.d./n.e.) por descumprimento de prazos contabeis (DCA/RGF/RREO)."},
    {"tipo": "limitacao", "item": "A nota CAPAG reflete a situacao fiscal global do municipio, nao a capacidade executiva da Defesa Civil local."},
    {"tipo": "regra_de_negocio", "item": "Priorizacao de Repasses: Municipios com CAPAG C e D devem ter prioridade absoluta no recurso a fundo perdido."},
    {"tipo": "regra_de_negocio", "item": "Impedimento Legal: Prefeituras C e D sao proibidas por lei de contratar financiamentos com garantia da Uniao (ex.: BNDES/Fundo Clima)."},
    {"tipo": "regra_de_negocio", "item": "Fundamentacao para Carlos (SEDEC): O repasse nao-reembolsavel e a unica alternativa legal e financeira para viabilizar obras nessas localidades."}
])

# Gerar o arquivo Excel com 3 abas
excel_filename = "DataCard_CAPAG_2025_Oficial.xlsx"
with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
    df_resumo.to_excel(writer, sheet_name="01_Resumo_Geral", index=False)
    df_distribuicao.to_excel(writer, sheet_name="02_Distribuição_Notas", index=False)
    df_limitacoes.to_excel(writer, sheet_name="03_Limitacoes_e_Regras", index=False)

# Gerar o arquivo ZIP com CSVs
zip_filename = "DataCard_CAPAG_2025_CSV.zip"
with zipfile.ZipFile(zip_filename, 'w') as z:
    z.writestr("01_resumo_geral.csv", df_resumo.to_csv(index=False, encoding='utf-8-sig'))
    z.writestr("02_distribuicao_notas.csv", df_distribuicao.to_csv(index=False, encoding='utf-8-sig'))
    z.writestr("03_limitacoes_recomendacoes.csv", df_limitacoes.to_csv(index=False, encoding='utf-8-sig'))

# Forçar o download automático no seu navegador
files.download(excel_filename)
files.download(zip_filename)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Expontando para PDF

In [21]:
!pip install reportlab

from google.colab import files
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib import colors
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle

pdf_filename = "Canvas_DataCard_CAPAG_2025.pdf"
doc = SimpleDocTemplate(
    pdf_filename,
    pagesize=landscape(letter),
    rightMargin=20, leftMargin=20, topMargin=20, bottomMargin=20
)

styles = getSampleStyleSheet()

# Estilos de Texto
title_style = ParagraphStyle('DocTitle', parent=styles['Heading1'], fontName='Helvetica-Bold', fontSize=18, leading=22, textColor=colors.HexColor('#1A365D'))
subtitle_style = ParagraphStyle('DocSubTitle', parent=styles['Normal'], fontName='Helvetica', fontSize=10, leading=12, textColor=colors.HexColor('#4A5568'))
header_box_style = ParagraphStyle('BoxHeader', parent=styles['Heading2'], fontName='Helvetica-Bold', fontSize=11, leading=13, textColor=colors.white)
body_bold_style = ParagraphStyle('BodyBold', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8.5, leading=10.5, textColor=colors.HexColor('#1A202C'))
body_style = ParagraphStyle('BodyTextCustom', parent=styles['Normal'], fontName='Helvetica', fontSize=8, leading=10, textColor=colors.HexColor('#2D3748'))
tag_c_d = ParagraphStyle('TagCD', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8, leading=10, textColor=colors.HexColor('#C53030'))
tag_a_b = ParagraphStyle('TagAB', parent=styles['Normal'], fontName='Helvetica-Bold', fontSize=8, leading=10, textColor=colors.HexColor('#276749'))

# 1. CONTEÚDO RESUMO GERAL
p1_content = [
    Paragraph("<b>Origem do Dado:</b> STN / Siconfi (Tesouro Nacional)", body_style),
    Paragraph("<b>Ano-Base / Versão:</b> Prévia Ano-Base 2025 (Set/2026)", body_style),
    Paragraph("<b>Granularidade:</b> 1 linha por Município (Brasil)", body_style),
    Paragraph("<b>Chave Primária:</b> Código IBGE de 7 dígitos", body_style),
    Paragraph("<b>Total de Municípios:</b> 5.568 prefeituras", body_style),
    Spacer(1, 4),
    Paragraph("<b>Qualidade & Preenchimento:</b>", body_bold_style),
    Paragraph("• <b>Nota Atribuída (A+ a D):</b> 4.747 mun. (85,26%)", body_style),
    Paragraph("• <b>Sem Nota (n.d. / n.e.):</b> 820 mun. (14,74%)", body_style),
    Paragraph("• <b>Inconsistências/Nulos na Chave:</b> 0 (Chave IBGE limpa)", body_style),
]

# 2. CONTEÚDO TABELA DISTRIBUIÇÃO
table_data = [
    [Paragraph("<b>Nota</b>", body_bold_style), Paragraph("<b>Perfil Fiscal</b>", body_bold_style), Paragraph("<b>Qtd</b>", body_bold_style), Paragraph("<b>%</b>", body_bold_style)],
    [Paragraph("C", tag_c_d), Paragraph("Fragilidade Fiscal / Vedado Crédito", body_style), Paragraph("2.184", body_style), Paragraph("39,22%", body_style)],
    [Paragraph("A", tag_a_b), Paragraph("Boa Saúde Fiscal / Elegível Crédito", body_style), Paragraph("1.190", body_style), Paragraph("21,37%", body_style)],
    [Paragraph("B", tag_a_b), Paragraph("Boa Saúde Fiscal / Elegível Crédito", body_style), Paragraph("976", body_style), Paragraph("17,53%", body_style)],
    [Paragraph("n.d.", body_style), Paragraph("Pendência Contábil (DCA/RGF)", body_style), Paragraph("683", body_style), Paragraph("12,27%", body_style)],
    [Paragraph("A+", tag_a_b), Paragraph("Excelente Saúde Fiscal", body_style), Paragraph("265", body_style), Paragraph("4,76%", body_style)],
    [Paragraph("n.e.", body_style), Paragraph("Não Elegível (Pendência STN)", body_style), Paragraph("137", body_style), Paragraph("2,46%", body_style)],
    [Paragraph("B+", tag_a_b), Paragraph("Boa Saúde Fiscal / Elegível Crédito", body_style), Paragraph("113", body_style), Paragraph("2,03%", body_style)],
    [Paragraph("D", tag_c_d), Paragraph("Alta Fragilidade Fiscal / Vedado Crédito", body_style), Paragraph("19", body_style), Paragraph("0,34%", body_style)],
]
t_dist = Table(table_data, colWidths=[28, 145, 32, 35])
t_dist.setStyle(TableStyle([
    ('BACKGROUND', (0,0), (-1,0), colors.HexColor('#EDF2F7')),
    ('GRID', (0,0), (-1,-1), 0.5, colors.HexColor('#CBD5E0')),
    ('TOPPADDING', (0,0), (-1,-1), 2), ('BOTTOMPADDING', (0,0), (-1,-1), 2),
]))
p2_content = [t_dist]

# 3. CONTEÚDO LIMITAÇÕES E RECOMENDAÇÕES
p3_content = [
    Paragraph("<b>1. Limitações dos Dados:</b>", body_bold_style),
    Paragraph("• <b>820 municípios (14,74%)</b> figuram como <i>n.d.</i> ou <i>n.e.</i> por descumprimento de prazos no envio das declarações contábeis (DCA/RGF/RREO) ao Siconfi/STN.", body_style),
    Paragraph("• A nota CAPAG reflete o risco fiscal global do ente municipal, e não a capacidade técnica/operacional da Defesa Civil local.", body_style),
    Spacer(1, 4),
    Paragraph("<b>2. Regra de Negócio & Recomendação (Carlos):</b>", body_bold_style),
    Paragraph("• <b>Prioridade Absoluta (CAPAG C e D):</b> Os <b>2.203 municípios (39,57%)</b> com notas C e D possuem impedimento legal de obter garantia da União para empréstimos bancários (ex.: BNDES / Fundo Clima).", body_style),
    Paragraph("• <b>Justificativa Técnica:</b> O repasse a fundo perdido (Sedec/MIDR) é a <b>única alternativa financeira legal</b> para viabilizar obras de prevenção contra inundações nessas cidades.", body_style),
]

# Função para construir as caixas do Canvas
def make_box(header_title, content_list, header_bg_color, width):
    header_table = Table([[Paragraph(header_title, header_box_style)]], colWidths=[width])
    header_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor(header_bg_color)),
        ('TOPPADDING', (0,0), (-1,-1), 5), ('BOTTOMPADDING', (0,0), (-1,-1), 5), ('LEFTPADDING', (0,0), (-1,-1), 8),
    ]))
    body_table = Table([[content_list]], colWidths=[width])
    body_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#FAFAFA')),
        ('BOX', (0,0), (-1,-1), 1, colors.HexColor('#E2E8F0')),
        ('TOPPADDING', (0,0), (-1,-1), 8), ('BOTTOMPADDING', (0,0), (-1,-1), 8),
        ('LEFTPADDING', (0,0), (-1,-1), 8), ('RIGHTPADDING', (0,0), (-1,-1), 8),
    ]))
    return Table([[header_table], [body_table]], colWidths=[width])

# Montagem do Grid Horizontal do Canvas
col1 = make_box("1. RESUMO GERAL", p1_content, "#2B6CB0", 240)
col2 = make_box("2. DISTRIBUIÇÃO DA CAPAG", p2_content, "#2C7A7B", 250)
col3 = make_box("3. LIMITAÇÕES E RECOMENDAÇÕES", p3_content, "#C53030", 250)

canvas_grid = Table([[col1, col2, col3]], colWidths=[245, 255, 255])
canvas_grid.setStyle(TableStyle([('VALIGN', (0,0), (-1,-1), 'TOP')]))

elements = [
    Paragraph("Canvas do Data Card — Capacidade de Pagamento (CAPAG 2025)", title_style),
    Paragraph("<b>Projeto:</b> Observatório do Clima | <b>Sprint 2 — Vaga 2.2</b> | <b>Persona:</b> Carlos (SEDEC/MIDR)", subtitle_style),
    Spacer(1, 8),
    HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#CBD5E0'), spaceAfter=10),
    canvas_grid
]

doc.build(elements)
files.download(pdf_filename)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.0 MB/s eta 0:00:00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>